# Section 1 : Environment Setup & Imports
Installing the required dependencies and load API keys.

## 1.1 installing dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

# !pip install -r requirements.txt

## 1.2 Loading the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Section 2 : Data Chunking
loading the mock data and split it into chunks.

In [3]:
from rag_utils import DataProcessor

# Initialize the Data processor
processor = DataProcessor()

# Load and split the document
chunks = processor.load_and_split(directory_path="data/", chunk_size=800, chunk_overlap=120)

print(f"Total chunks created: {len(chunks)}")

sources = set([chunk.metadata.get("source") for chunk in chunks])
print(f"Sources loaded: {sources}")

100%|██████████| 1/1 [00:00<00:00, 427.51it/s]

Total chunks created: 13
Sources loaded: {'data/github_projects.md', 'data/certifications.txt', 'data/Mohammad_CV.pdf', 'data/linkedin_profile.txt'}


# Section 3 : Vector Database & Retriever
Convert the text chunks into embeddings using HuggingFace's `all-MiniLM-L6-v2` model and store them in a ChromaDB vector store.

In [4]:
# Create vector store
vectorstore = processor.create_vector_store(chunks, persist_directory="./chroma_db")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# Section 4 : Building the RAG Pipeline
inject our retriever into the `RAGPipeline` class. This class handles the prompt engineering and the LangChain Expression Language (LCEL) to ensure the LLM answers based *only* on the retrieved context and cites its sources.

In [5]:
from rag_chain import RAGPipeline

# Initialize the RAG pipeline with the retriever
rag_pipeline = RAGPipeline(retriever=retriever)
rag_chain = rag_pipeline.build()

# Section 5 : Inference

In [14]:
from IPython.display import display, Markdown, HTML

def query_rag(question: str):
    result = rag_chain.invoke(question)

    # question
    display(Markdown(f"## {question}"))
    # answer
    display(Markdown(result['answer']))

    display(Markdown("---"))

    # Sources
    sources_html = "<details><summary><b> View source documents</b></summary>\n\n"
    for i, doc in enumerate(result["context"]):
        source_name = doc.metadata.get("source", "Unknown")
        sources_html += f"<p><b>[{i+1}] {source_name}</b><br><pre>{doc.page_content.strip()}</pre></p>\n"
    sources_html += "</details>"
    display(HTML(sources_html))

## Test 1: Specific skill question

In [15]:
query_rag("What are Mohammad's main programming languages?")

## What are Mohammad's main programming languages?

Mohammad's main programming languages are Python, Swift, C++, Java, and Dart.

---

## Test 2: Experience question

In [16]:
query_rag("Where did Mohammad work and what did he do there?")

## Where did Mohammad work and what did he do there?

Mohammad has worked at two companies:

1. **Cellula Technologies – AI Engineer Intern** (September 2026 – Present, Remote)  
   - Worked on practical AI problems using real‑world datasets.  
   - Contributed to LLM and retrieval‑augmented generation (RAG) systems, voice‑enabled AI agents, and computer‑vision models.  
   - Built end‑to‑end AI pipelines with Docker and CI/CD, collaborating with engineering teams on deployment, debugging, and scaling models.

2. **ASAL Technologies – Mobile Development Intern** (October 2024 – April 2025, Rawabi, Palestine)  
   - Re‑engineered the application architecture to improve stability, scalability, and performance across core workflows.  
   - Developed interactive community features and real‑time messaging.  
   - Ensured reliability through comprehensive testing and debugging.

---

## Test 3: GitHub question

In [17]:
query_rag("What kind of projects has Mohammad worked on?")

## What kind of projects has Mohammad worked on?

Mohammad has worked on a variety of projects that span both AI‑focused and mobile‑development domains:

- **AI Engineer Intern – Cellula Technologies**  
  - Built production‑grade AI solutions, including large language models (LLMs), retrieval‑augmented generation (RAG) systems, voice‑enabled AI agents, and computer‑vision models using real‑world datasets.  
  - Developed end‑to‑end AI pipelines with Docker and CI/CD, collaborating on deployment, debugging, and scaling of models.

- **Mobile Development Intern – ASAL Technologies**  
  - Re‑engineered the application architecture to improve stability, scalability, and performance across core workflows.  
  - Created interactive community features and real‑time messaging functionality, ensuring reliability through comprehensive testing and debugging.

- **Personal/Project Work – Plantie App**  
  - Architected a full‑stack mobile application using Flutter and the BLoC state‑management pattern, integrating AI capabilities into the app’s architecture.

These projects demonstrate his experience in developing AI‑driven systems (LLMs, RAG, voice agents, computer vision) as well as designing and implementing scalable mobile applications with real‑time features and robust architecture.

---

## Test 4: Certifications question

In [18]:
query_rag("What certifications does Mohammad have?")

## What certifications does Mohammad have?

Mohammad has the following certifications:

- **PIP Professional Development Curriculum** – Palestinian Internship Program (completed 2024, six‑month program; Credential ID: PIP‑PDC‑2024‑0822)  
- **The Complete 2022 Flutter & Dart Development Course** – Udemy, instructed by Abdullah Mansour (completed 18 June 2022, 42 hours; Credential ID: UD‑FLT‑2022‑0618)  
- **Native iOS Development Training** – ASAL Technologies (completed 2025; Credential ID: ASAL‑IOS‑2025‑0703)  
- **Data Science & Machine Learning** – IBT Learning Solutions and GGateway (January 2026 – April 2026, three‑month program; Credential ID: IBT‑DSML‑2026‑0411)  
- **Comprehensive Artificial Intelligence Training** – Independent AI training program (November 2024 – January 2025, 48 hours; Credential ID: AI‑TRN‑2025‑0115)

---

## Test 5: Out-of-scope question (testing hallucination guardrails)

In [19]:
query_rag("What is Mohammad's favorite food?")

## What is Mohammad's favorite food?

I don't have enough information to answer that.

---

## Test 6: CV PDF Source

In [21]:
query_rag("What is Mohammad's summary in his cv ?")

## What is Mohammad's summary in his cv ?

Computer Engineer with a strong foundation in software engineering and problem solving skills. Specialized in mobile development and AI, with expertise in driving the complete machine learning pipeline. Adept at seamlessly integrating models into end-to-end applications, whether through optimized local on‑device inference or by scalable APIs. Thrives in dynamic technical environments and quickly adapts to new tech stacks, with a continuous focus on delivering reliable, high‑impact software solutions.

---

In [20]:
query_rag("What is Mohammad's email ?")

## What is Mohammad's email ?

m.nassar4004@gmail.com

---